In [1]:
from __future__ import annotations
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import csv
import polars as pl
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.table import Table, TableStyleInfo

BASE = Path(
    r"J:\Product & Client Solutions\Toolbox\Calculators"
    r"\Business Tools\Tender Summary\GAP Pricing Engine"
)
SOURCE       = BASE / "Basis and Assumptions.xlsx"
MASTER_XLSX  = BASE / "basis_master" / "basis_master_v2026_07.xlsx"
SNAPSHOT_DIR = BASE / "basis_snapshots" / "basis_v2026_07"
BUILD_LOG    = SNAPSHOT_DIR / "build_log.csv"

MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
assert SOURCE.exists()
print("✅ Setup ready")

✅ Setup ready


In [2]:
def find_anchors(ws, text: str) -> list[tuple[int, int]]:
    hits = []
    for row in ws.iter_rows():
        for cell in row:
            v = cell.value
            if isinstance(v, str) and v.strip() == text:
                hits.append((cell.row, cell.column))
    return hits


def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

print("✅ Helpers ready")

✅ Helpers ready


In [3]:
print(f"📂 Opening source: {SOURCE.name}\n")
wb = openpyxl.load_workbook(SOURCE, data_only=True)

# ---------- GLA Basis: find the industry class table ----------
print("🔎 GLA Basis — industry class table")
ws = wb["GLA Basis"]
for anchor in ["Industry Class", "Industry Grouping", "Grouping for experience investigation"]:
    hits = find_anchors(ws, anchor)
    if hits:
        print(f"  Anchor '{anchor}' → {len(hits)} occurrence(s):")
        for r, c in hits[:3]:
            preview = [(ws.cell(row=r+i, column=c).value,
                        ws.cell(row=r+i, column=c+1).value,
                        ws.cell(row=r+i, column=c+2).value,
                        ws.cell(row=r+i, column=c+3).value,
                        ws.cell(row=r+i, column=c+4).value) for i in range(1, 4)]
            print(f"    row={r}, col={c}, first 3 data rows: {preview}")

# ---------- LSDB Basis: find the industry class table ----------
print("\n🔎 LSDB Basis — industry class table")
ws = wb["LSDB Basis"]
for anchor in ["Ind Class", "Industry Grouping", "Industry Class"]:
    hits = find_anchors(ws, anchor)
    if hits:
        print(f"  Anchor '{anchor}' → {len(hits)} occurrence(s):")
        for r, c in hits[:3]:
            preview = [(ws.cell(row=r+i, column=c).value,
                        ws.cell(row=r+i, column=c+1).value,
                        ws.cell(row=r+i, column=c+2).value) for i in range(1, 4)]
            print(f"    row={r}, col={c}, first 3 data rows: {preview}")

# ---------- CI Basis: find class A-E tables + hazard ----------
print("\n🔎 CI Basis — class A-E tables + hazard occupations")
ws = wb["CI Basis"]
for anchor in ["Industry", "Adjustment", "Loading", "Industry Adjustment", "Industry factors"]:
    hits = find_anchors(ws, anchor)
    if hits:
        print(f"  Anchor '{anchor}' → {len(hits)} occurrence(s):")
        for r, c in hits[:6]:
            preview = [(ws.cell(row=r+i, column=c).value,
                        ws.cell(row=r+i, column=c+1).value) for i in range(1, 4)]
            print(f"    row={r}, col={c}, first 3 rows: {preview}")

# ---------- Spouse Basis: find hazard ----------
print("\n🔎 Spouse Basis — industry hazard table")
ws = wb["Spouse Basis"]
for anchor in ["Industry factors", "Industry", "Underground mining"]:
    hits = find_anchors(ws, anchor)
    if hits:
        print(f"  Anchor '{anchor}' → {len(hits)} occurrence(s):")
        for r, c in hits[:3]:
            preview = [(ws.cell(row=r+i, column=c).value,
                        ws.cell(row=r+i, column=c+1).value) for i in range(1, 5)]
            print(f"    row={r}, col={c}, first 4 rows: {preview}")

print("\n✅ Diagnostic complete")

📂 Opening source: Basis and Assumptions.xlsx

🔎 GLA Basis — industry class table
  Anchor 'Industry Class' → 1 occurrence(s):
    row=4, col=23, first 3 data rows: [('Mining, Resource Extraction & Quarrying ', 'Mining, Resource Extraction & Quarrying', 1.7, 1.05, 1), ('Municipalities', 'Municipalities', 1.15, 1.5, 1.1), ('Agriculture - Farming, Hunting and Related Activities', 'Agriculture - Farming, Hunting and Related Activities', 1, 1, 1)]
  Anchor 'Industry Grouping' → 1 occurrence(s):
    row=4, col=24, first 3 data rows: [('Mining, Resource Extraction & Quarrying', 1.7, 1.05, 1, 1.15), ('Municipalities', 1.15, 1.5, 1.1, 1), ('Agriculture - Farming, Hunting and Related Activities', 1, 1, 1, 1)]
  Anchor 'Grouping for experience investigation' → 1 occurrence(s):
    row=89, col=5, first 3 data rows: [('Agriculture - Farming, Hunting and Related Activities', 1, 1, 1, 1), ('Fishing and Aquaculture', 2, 1.075, 1.075, 1), ('Forestry and Logging', 3, 1.075, 1.075, 1)]

🔎 LSDB Basis — in

In [4]:
print(f"📂 Opening source: {SOURCE.name}\n")
wb = openpyxl.load_workbook(SOURCE, data_only=True)

all_rows = []

# ============================================================
# 1. GLA industry class table
# Anchor: "Industry Class" at (4, 23)
# Cols from anchor: 0=grouping, 1=grouping_dup, 2=small_inc, 3=large_inc, 4=small_acaf, 5=large_acaf
# ============================================================
print("🔎 GLA industry class (small + large scheme)...")
ws = wb["GLA Basis"]
anchors = find_anchors(ws, "Industry Class")
r0, c0 = anchors[0]
r = r0 + 1
gla_count = 0
while r < 200:
    grouping = ws.cell(row=r, column=c0).value
    if grouping is None:
        break
    grouping = str(grouping).strip()
    small_inc  = ws.cell(row=r, column=c0+2).value
    large_inc  = ws.cell(row=r, column=c0+3).value
    small_acaf = ws.cell(row=r, column=c0+4).value
    large_acaf = ws.cell(row=r, column=c0+5).value

    if small_inc is None and large_inc is None:
        break

    all_rows.append(dict(product="gla", grouping=grouping, scheme_size="small",
                         inc_adj=float(small_inc)  if small_inc  is not None else None,
                         acaf_adj=float(small_acaf) if small_acaf is not None else None))
    all_rows.append(dict(product="gla", grouping=grouping, scheme_size="large",
                         inc_adj=float(large_inc)  if large_inc  is not None else None,
                         acaf_adj=float(large_acaf) if large_acaf is not None else None))
    gla_count += 1
    r += 1
print(f"  ✅ gla   → {gla_count} groupings × 2 scheme_sizes = {gla_count*2} rows")

# ============================================================
# 2. LSDB industry class table
# Anchor: "Ind Class" at (18, 8)
# Cols from anchor: 0=grouping, 1=grouping_dup, 2=inc_adj, 3=acaf_adj
# ============================================================
print("\n🔎 LSDB industry class...")
ws = wb["LSDB Basis"]
anchors = find_anchors(ws, "Ind Class")
r0, c0 = anchors[0]
r = r0 + 1
lsdb_count = 0
while r < 100:
    grouping = ws.cell(row=r, column=c0).value
    if grouping is None:
        break
    grouping = str(grouping).strip()
    inc  = ws.cell(row=r, column=c0+2).value
    acaf = ws.cell(row=r, column=c0+3).value
    if inc is None:
        break
    all_rows.append(dict(product="lsdb", grouping=grouping, scheme_size=None,
                         inc_adj=float(inc),
                         acaf_adj=float(acaf) if acaf is not None else None))
    lsdb_count += 1
    r += 1
print(f"  ✅ lsdb  → {lsdb_count} groupings")

# ============================================================
# 3. CI A-E class tables — Old and New
# Anchor: "Industry" at (153, 15) [Old] and (153, 19) [New]
# Each: 5 rows for A, B, C, D, E
# ============================================================
print("\n🔎 CI A-E class tables (Old + New)...")
ws = wb["CI Basis"]
industry_anchors = [(r, c) for (r, c) in find_anchors(ws, "Industry") if r == 153]
# Sort by column so Old (col 15) comes first, then New (col 19)
industry_anchors.sort(key=lambda rc: rc[1])

if len(industry_anchors) >= 2:
    labels = ["ci_old", "ci_new"]
    for (r0, c0), product_key in zip(industry_anchors, labels):
        r = r0 + 1
        ci_count = 0
        for _ in range(5):
            cls = ws.cell(row=r, column=c0).value
            fac = ws.cell(row=r, column=c0+1).value
            if cls is None or fac is None:
                break
            all_rows.append(dict(product=product_key, grouping=str(cls).strip(),
                                 scheme_size=None,
                                 inc_adj=float(fac), acaf_adj=None))
            ci_count += 1
            r += 1
        print(f"  ✅ {product_key} → {ci_count} classes")
else:
    print(f"  ❌ Expected 2 CI class tables at row 153, found {len(industry_anchors)}")

# ============================================================
# 4. CI hazard occupation table
# Anchor: "Industry factors" at (68, 16)
# 3 rows: Underground mining / Longdistance transport / Other
# Apply to ci_acc AND ci_std
# ============================================================
print("\n🔎 CI hazard occupations (→ ci_acc + ci_std)...")
anchors = find_anchors(ws, "Industry factors")
if anchors:
    r0, c0 = anchors[0]
    r = r0 + 1
    hazard_rows = []
    for _ in range(5):
        g   = ws.cell(row=r, column=c0).value
        fac = ws.cell(row=r, column=c0+1).value
        if g is None or fac is None:
            break
        hazard_rows.append((str(g).strip(), float(fac)))
        r += 1
    for product_key in ["ci_acc", "ci_std"]:
        for g, fac in hazard_rows:
            all_rows.append(dict(product=product_key, grouping=g, scheme_size=None,
                                 inc_adj=fac, acaf_adj=None))
    print(f"  ✅ ci_acc + ci_std → {len(hazard_rows)} groupings each = {len(hazard_rows)*2} rows")
else:
    print(f"  ❌ CI hazard 'Industry factors' anchor not found")

# ============================================================
# 5. Spouse hazard occupation table
# Anchor: "Industry factors" at (2, 17) on Spouse Basis
# 3 rows: Underground mining / Longdistance transport / Other
# Apply to spouse
# ============================================================
print("\n🔎 Spouse hazard occupations (→ spouse)...")
ws = wb["Spouse Basis"]
anchors = find_anchors(ws, "Industry factors")
if anchors:
    r0, c0 = anchors[0]
    r = r0 + 1
    sp_count = 0
    for _ in range(5):
        g   = ws.cell(row=r, column=c0).value
        fac = ws.cell(row=r, column=c0+1).value
        if g is None or fac is None:
            break
        all_rows.append(dict(product="spouse", grouping=str(g).strip(),
                             scheme_size=None,
                             inc_adj=float(fac), acaf_adj=None))
        sp_count += 1
        r += 1
    print(f"  ✅ spouse → {sp_count} groupings")
else:
    print(f"  ❌ Spouse hazard 'Industry factors' anchor not found")

# ============================================================
# Assemble
# ============================================================
df = pl.DataFrame(all_rows).sort(["product", "grouping"])
print(f"\n📊 Total rows: {df.height}  |  Distinct products: {df['product'].n_unique()}")
df.group_by("product").agg(pl.len().alias("n")).sort("product")


📂 Opening source: Basis and Assumptions.xlsx

🔎 GLA industry class (small + large scheme)...
  ✅ gla   → 28 groupings × 2 scheme_sizes = 56 rows

🔎 LSDB industry class...
  ✅ lsdb  → 25 groupings

🔎 CI A-E class tables (Old + New)...
  ✅ ci_old → 5 classes
  ✅ ci_new → 5 classes

🔎 CI hazard occupations (→ ci_acc + ci_std)...
  ✅ ci_acc + ci_std → 3 groupings each = 6 rows

🔎 Spouse hazard occupations (→ spouse)...
  ✅ spouse → 3 groupings

📊 Total rows: 100  |  Distinct products: 7


product,n
str,u32
"""ci_acc""",3
"""ci_new""",5
"""ci_old""",5
"""ci_std""",3
"""gla""",56
"""lsdb""",25
"""spouse""",3


In [5]:
SANITY_CHECKS = [
    # (product, grouping, scheme_size, expected_inc_adj, expected_acaf_or_None)
    ("gla",     "Mining, Resource Extraction & Quarrying", "small", 1.70, 1.00),
    ("gla",     "Mining, Resource Extraction & Quarrying", "large", 1.05, 1.15),
    ("gla",     "Education",                                "small", 0.85, 0.95),
    ("gla",     "Municipalities",                           "small", 1.15, 1.10),
    ("gla",     "Municipalities",                           "large", 1.50, 1.00),
    ("lsdb",    "Agriculture, Forestry and Fishing",        None,    0.95, 0.90),
    ("lsdb",    "Mining, Resource Extraction & Quarrying",  None,    1.45, 1.00),
    ("ci_old",  "A",                                        None,    0.6666666666666669, None),
    ("ci_old",  "E",                                        None,    0.96, None),
    ("ci_new",  "A",                                        None,    0.8333333333333335, None),
    ("ci_new",  "E",                                        None,    1.20, None),
    ("ci_acc",  "Underground mining",                       None,    2.23, None),
    ("ci_std",  "Longdistance transport",                   None,    2.23, None),
    ("spouse",  "Other",                                    None,    1.00, None),
]

print("🧪 Sanity checks:\n")
all_pass = True
for product, grouping, scheme_size, exp_inc, exp_acaf in SANITY_CHECKS:
    if scheme_size is None:
        hit = df.filter((pl.col("product") == product) &
                        (pl.col("grouping") == grouping) &
                        (pl.col("scheme_size").is_null()))
    else:
        hit = df.filter((pl.col("product") == product) &
                        (pl.col("grouping") == grouping) &
                        (pl.col("scheme_size") == scheme_size))
    if hit.is_empty():
        print(f"  ❌ NOT FOUND  {product:<10} '{grouping[:40]}' scheme={scheme_size}")
        all_pass = False
        continue
    row = hit.row(0, named=True)
    inc_ok  = abs((row["inc_adj"] or 0) - exp_inc) < 1e-6
    acaf_ok = (exp_acaf is None) or (row["acaf_adj"] is not None
                                     and abs(row["acaf_adj"] - exp_acaf) < 1e-6)
    status = "✅ MATCH" if (inc_ok and acaf_ok) else "❌ MISMATCH"
    if not (inc_ok and acaf_ok):
        all_pass = False
    print(f"  {status}  {product:<10} '{grouping[:40]:<40}' "
          f"scheme={str(scheme_size):<7} inc={row['inc_adj']}  acaf={row['acaf_adj']}")

print(f"\n{'🎉 ALL PASSED' if all_pass else '⚠️ FIX FAILURES BEFORE WRITING'}")


🧪 Sanity checks:

  ✅ MATCH  gla        'Mining, Resource Extraction & Quarrying ' scheme=small   inc=1.7  acaf=1.0
  ✅ MATCH  gla        'Mining, Resource Extraction & Quarrying ' scheme=large   inc=1.05  acaf=1.15
  ✅ MATCH  gla        'Education                               ' scheme=small   inc=0.85  acaf=0.95
  ✅ MATCH  gla        'Municipalities                          ' scheme=small   inc=1.15  acaf=1.1
  ✅ MATCH  gla        'Municipalities                          ' scheme=large   inc=1.5  acaf=1.0
  ✅ MATCH  lsdb       'Agriculture, Forestry and Fishing       ' scheme=None    inc=0.95  acaf=0.9
  ✅ MATCH  lsdb       'Mining, Resource Extraction & Quarrying ' scheme=None    inc=1.45  acaf=1.0
  ✅ MATCH  ci_old     'A                                       ' scheme=None    inc=0.6666666666666669  acaf=None
  ✅ MATCH  ci_old     'E                                       ' scheme=None    inc=0.96  acaf=None
  ✅ MATCH  ci_new     'A                                       ' scheme=Non

In [6]:
# ---------- Guard: Excel file lock ----------
if MASTER_XLSX.exists():
    try:
        with open(MASTER_XLSX, "a"):
            pass
    except PermissionError:
        raise PermissionError(
            f"❌ '{MASTER_XLSX.name}' is open in Excel. Close it and re-run.")

# ---------- 1. Parquet snapshot ----------
parquet_path = SNAPSHOT_DIR / "industry_factors.parquet"
df.write_parquet(parquet_path)
parquet_size = parquet_path.stat().st_size / 1024
parquet_hash = sha256_of(parquet_path)
print(f"📦 Parquet: {parquet_path.name}  ({parquet_size:.1f} KB)")

# ---------- 2. Master xlsx sheet ----------
if MASTER_XLSX.exists():
    wb_out = openpyxl.load_workbook(MASTER_XLSX)
    if "industry_factors" in wb_out.sheetnames:
        del wb_out["industry_factors"]
else:
    wb_out = openpyxl.Workbook()
    if "Sheet" in wb_out.sheetnames:
        del wb_out["Sheet"]

ws = wb_out.create_sheet("industry_factors")
headers = ["product", "grouping", "scheme_size", "inc_adj", "acaf_adj"]
ws.append(headers)

for cell in ws[1]:
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = PatternFill("solid", fgColor="305496")
    cell.alignment = Alignment(horizontal="left")

for row in df.iter_rows():
    ws.append(row)

widths = {"A": 12, "B": 55, "C": 12, "D": 12, "E": 12}
for col, w in widths.items():
    ws.column_dimensions[col].width = w
ws.freeze_panes = "A2"

table = Table(displayName="tbl_industry_factors", ref=f"A1:E{ws.max_row}")
table.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
ws.add_table(table)
wb_out.save(MASTER_XLSX)
print(f"📄 Master xlsx: sheet 'industry_factors' → {df.height} rows")

# ---------- 3. Append build log ----------
source_hash = sha256_of(SOURCE)
timestamp   = datetime.now(timezone.utc).isoformat(timespec="seconds")

log_row = {
    "timestamp_utc":     timestamp,
    "sheet":             "industry_factors",
    "row_count":         df.height,
    "distinct_products": df["product"].n_unique(),
    "source_file":       SOURCE.name,
    "source_sha256":     source_hash,
    "parquet_file":      parquet_path.name,
    "parquet_sha256":    parquet_hash,
    "parquet_kb":        round(parquet_size, 2),
    "status":            "ok",
    "built_by":          "Mohamed Hussein Gadiwala",
}

log_exists = BUILD_LOG.exists()
with open(BUILD_LOG, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(log_row.keys()))
    if not log_exists:
        writer.writeheader()
    writer.writerow(log_row)

print(f"📝 Build log:  appended row to {BUILD_LOG.name}")
print(f"\n🎉 Done!")

📦 Parquet: industry_factors.parquet  (2.8 KB)
📄 Master xlsx: sheet 'industry_factors' → 100 rows
📝 Build log:  appended row to build_log.csv

🎉 Done!
